# Phase 1: CAISO 5-min LMP Exploration

**Goal**: Validate data quality, understand distributions, compute spike class frequencies, lock thresholds.

**Output**: Class frequency tables, threshold decisions, schema notes.

**Data**: 7-day sample from `data/processed/caiso_lmp_7d_sample.parquet` (5,799 rows × 11 cols, 247 KB).

**Status**: Phase 1 complete. Spike thresholds locked at 1.5x/3x/6x with 4h baseline. See `docs/PHASE1_FINDINGS.md` for full empirical report.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load 7-day sample
df = pd.read_parquet('/root/project/dc_real_time/data/processed/caiso_lmp_7d_sample.parquet')
print(f"Loaded {df.shape[0]} rows × {df.shape[1]} cols")
print(df.head(3))

In [ ]:
# Per-zone LMP stats
print("\n=== LMP stats per zone (7 days) ===")
print(df.groupby('Location')['LMP'].describe())

In [ ]:
# Multi-class spike label function
def label_spike(lmp_series, window='4h', thresholds=(1.5, 3.0, 6.0)):
    baseline = lmp_series.rolling(window).mean()
    ratio = lmp_series / baseline
    labels = pd.cut(ratio, bins=[0, *thresholds, float('inf')], labels=[0,1,2,3], right=False)
    return labels.astype('Int64')

# Apply per zone
print("\n=== Spike class frequency (4h baseline, 1.5x/3x/6x) ===")
for loc, sub in df.groupby('Location'):
    sub = sub.sort_values('Time').set_index('Time')
    labels = label_spike(sub['LMP'])
    total = len(labels.dropna())
    print(f"\n{loc}:")
    for cls, name in {0:'Normal',1:'Moderate',2:'High',3:'Extreme'}.items():
        n = (labels == cls).sum()
        print(f"  Class {cls} ({name:8s}): {n:5d} ({n/total*100:5.2f}%)"

In [ ]:
# Visualization: 24h LMP trace per zone
fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for i, (loc, sub) in enumerate(df.groupby('Location')):
    sub = sub.sort_values('Time')
    ax[i].plot(sub['Time'], sub['LMP'], linewidth=0.5)
    ax[i].set_title(f'{loc} 5-min LMP (7d)')
    ax[i].set_ylabel('LMP ($/MWh)')
    ax[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/root/project/dc_real_time/artifacts/lmp_7d_trace.png', dpi=100)
plt.show()

## Findings Summary

| Zone | Mean LMP | Std LMP | Max LMP | Class 3 (Extreme) % |
|------|----------|---------|---------|--------------------|
| NP15 | $14.51 | $13.15 | $126.93 | 1.48% |
| SP15 | $9.21 | $12.52 | $39.93 | 2.44% |
| ZP26 | $11.06 | $12.95 | $68.95 | 2.27% |

**Decisions locked** (see DECISIONS.md):
- D1: Multi-class spike classifier with 4 levels
- D8: Thresholds = 1.5x / 3.0x / 6.0x with 4h rolling baseline
- D9: Use `caiso.get_lmp(date=...)` for backfill (range mode broken in OASIS)
- D10: GHG (carbon intensity) is system-wide, not per-zone

**Next steps**:
- Run `notebooks/colab_handoff/01_caiso_1y_backfill.md` on Colab Pro for 1y history
- Then: Phase 2 (feature engineering), Phase 3 (XGBoost baseline)